# Financial Time Series Data Collection and Preprocessing for Bayesian Causal Modeling

This notebook demonstrates how to collect, clean, and preprocess financial time series data for modeling purposes. We focus on two key datasets:
- **Federal Funds Rate** (obtained from FRED)
- **S&P 500 Index** (obtained from Yahoo Finance)

We then perform data cleaning, resampling, and feature engineering to prepare stationary time series for further analysis (such as Bayesian VAR or Bayesian RNN models). Finally, we verify stationarity using the Augmented Dickey-Fuller (ADF) test.

## 1. Setup and Data Retrieval

In this section, we import the required libraries, and define the date range for data collection. We then download the Federal Funds Rate from FRED and S&P 500 data from Yahoo Finance.

In [69]:
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader import data as pdr
from statsmodels.tsa.stattools import adfuller

## Data Collection

In [3]:
# Defining the date range for our data (several decades) to account for different economic regimes
start_date = "1950-01-01"
end_date   = "2025-03-31"

In [19]:
#-- A. Federal Funds Rate from FRED -------------------
fedfunds_df = pdr.DataReader("FEDFUNDS", "fred", start_date, end_date)
# Convert from 1-column DataFrame into a Series
# (fedfunds_df.columns -> ['FEDFUNDS'])
fedfunds_s  = fedfunds_df['FEDFUNDS'].copy()

In [20]:
fedfunds_s

DATE
1954-07-01    0.80
1954-08-01    1.22
1954-09-01    1.07
1954-10-01    0.85
1954-11-01    0.83
              ... 
2024-10-01    4.83
2024-11-01    4.64
2024-12-01    4.48
2025-01-01    4.33
2025-02-01    4.33
Name: FEDFUNDS, Length: 848, dtype: float64

In [21]:
# By default, the Fed Funds FRED series is monthly, but
# often the index is the 1st of each month. Shift to end-of-month:
fedfunds_s.index = fedfunds_s.index + pd.offsets.MonthEnd(0)

In [22]:
fedfunds_s

DATE
1954-07-31    0.80
1954-08-31    1.22
1954-09-30    1.07
1954-10-31    0.85
1954-11-30    0.83
              ... 
2024-10-31    4.83
2024-11-30    4.64
2024-12-31    4.48
2025-01-31    4.33
2025-02-28    4.33
Name: FEDFUNDS, Length: 848, dtype: float64

In [36]:
#-- B. S&P 500 from yfinance (daily data) -------------
sp500_df = yf.download("^GSPC", start=start_date, end=end_date, progress=False)
# Some indexes don't have 'Adj Close', so use 'Close' safely:
# Confirm you have a 'Close' column:
if 'Close' not in sp500_df.columns:
    raise ValueError("Could not find a 'Close' column in the downloaded S&P 500 DataFrame!")

In [37]:
sp500_df.columns

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])

## 2. Data Cleaning and Resampling

Here we perform the following steps:
- **Extract and Resample Data:** We extract the relevant columns (e.g., 'Close' for the S&P 500) and resample the data to a monthly frequency (using month-end dates).
- **Align Time Series:** We ensure both datasets share a common time index by aligning them based on overlapping dates.
- **Handle Missing Values:** Any missing data points are forward-filled to maintain continuity.

In [38]:
# 1) Keep only the "Close" column
sp500_close_df = sp500_df[['Close']]

In [40]:
sp500_close_df.shape

(18930, 1)

In [41]:
# 2) Resample to month-end frequency
sp500_close_df = sp500_close_df.resample('ME').last()

In [43]:
# 3) Rename the column to "SP500"
sp500_close_df.columns = ['SP500']

In [44]:
print(sp500_close_df)

                  SP500
Date                   
1950-01-31    17.049999
1950-02-28    17.219999
1950-03-31    17.290001
1950-04-30    17.959999
1950-05-31    18.780001
...                 ...
2024-11-30  6032.379883
2024-12-31  5881.629883
2025-01-31  6040.529785
2025-02-28  5954.500000
2025-03-31  5580.939941

[903 rows x 1 columns]


In [45]:
# 4) Extract the single column as a Series
sp500_monthly_s = sp500_close_df['SP500']

In [46]:
sp500_monthly_s

Date
1950-01-31      17.049999
1950-02-28      17.219999
1950-03-31      17.290001
1950-04-30      17.959999
1950-05-31      18.780001
                 ...     
2024-11-30    6032.379883
2024-12-31    5881.629883
2025-01-31    6040.529785
2025-02-28    5954.500000
2025-03-31    5580.939941
Freq: ME, Name: SP500, Length: 903, dtype: float64

In [49]:
# 3. Combine the two series into one DataFrame by date
data = pd.concat([fedfunds_s, sp500_monthly_s], axis=1)
data.columns = ['FedFundsRate','SP500']

In [50]:
data

,FedFundsRate,SP500
1950-01-31,NaN,17.049999
1950-02-28,NaN,17.219999
1950-03-31,NaN,17.290001
1950-04-30,NaN,17.959999
1950-05-31,NaN,18.780001
...,...,...
2024-11-30,4.64,6032.379883
2024-12-31,4.48,5881.629883
2025-01-31,4.33,6040.529785
2025-02-28,4.33,5954.500000


In [51]:
# 4. Handle missing dates by forward-filling or dropping
data = data.sort_index()           # ensure sorted by date
data = data.loc['1957':]           # drop initial months with no S&P500 data (if any before 1957)

In [52]:
data

,FedFundsRate,SP500
1957-01-31,2.84,44.720001
1957-02-28,3.00,43.259998
1957-03-31,2.96,44.110001
1957-04-30,3.00,45.740002
1957-05-31,3.00,47.430000
...,...,...
2024-11-30,4.64,6032.379883
2024-12-31,4.48,5881.629883
2025-01-31,4.33,6040.529785
2025-02-28,4.33,5954.500000


In [54]:
data[data.isna().any(axis=1)]

,FedFundsRate,SP500
2025-03-31,NaN,5580.939941


In [55]:
data.isna().sum()

FedFundsRate    1
SP500           0
dtype: int64

In [56]:
data = data.fillna(method='ffill') # forward-fill any small gaps in between, if present

# 5. Remove any remaining NaNs (e.g. if trailing data missing)
data.dropna(inplace=True)

print(data.shape, data.index.freq)  # Check dimensions and frequency
print(data.head())                  # Preview the first few rows after alignment

(819, 2) <MonthEnd>
            FedFundsRate      SP500
1957-01-31          2.84  44.720001
1957-02-28          3.00  43.259998
1957-03-31          2.96  44.110001
1957-04-30          3.00  45.740002
1957-05-31          3.00  47.430000


/tmp/ipykernel_4146187/3641626550.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill') # forward-fill any small gaps in between, if present


## 3. Feature Engineering and Stationarity Transformations

To meet modeling assumptions, we transform our time series to induce stationarity:
- **S&P 500 Log Returns:** We calculate the monthly logarithmic returns.
- **Fed Funds Rate Differences:** We compute the first difference of the Federal Funds Rate.

These transformations help stabilize the mean and variance of the series, making them more suitable for time series modeling.

In [57]:
# Create stationarity-inducing transformations
data['SP500_log_return'] = np.log(data['SP500'] / data['SP500'].shift(1))
data['FedFunds_diff']    = data['FedFundsRate'].shift(0) - data['FedFundsRate'].shift(1)


In [58]:
# Drop the first row which will have NaN due to the shift
data.dropna(inplace=True)

In [59]:

print(data[['SP500_log_return','FedFunds_diff']].head())

            SP500_log_return  FedFunds_diff
1957-02-28         -0.033192           0.16
1957-03-31          0.019458          -0.04
1957-04-30          0.036287           0.04
1957-05-31          0.036282           0.00
1957-06-30         -0.001266           0.00


In [60]:
print(data.shape)

(818, 4)


## 4. Stationarity Verification

We verify the stationarity of our transformed series using the Augmented Dickey-Fuller (ADF) test. The null hypothesis of the ADF test is that the series is non-stationary (has a unit root). A p-value below 0.05 indicates we can reject the null hypothesis and treat the series as stationary.

In [70]:
# Check stationarity of S&P 500 log returns
adf_result_sp500 = adfuller(data['SP500_log_return'].dropna())
print("S&P 500 Log Returns - ADF Statistic: ", adf_result_sp500[0])
print("S&P 500 Log Returns - p-value: ", adf_result_sp500[1])

# Check stationarity of Fed Funds differences
adf_result_fedfunds = adfuller(data['FedFunds_diff'].dropna())
print("Fed Funds Diff - ADF Statistic: ", adf_result_fedfunds[0])
print("Fed Funds Diff - p-value: ", adf_result_fedfunds[1])

S&P 500 Log Returns - ADF Statistic:  -11.379110151957866
S&P 500 Log Returns - p-value:  8.598520171640382e-21
Fed Funds Diff - ADF Statistic:  -6.306097421325353
Fed Funds Diff - p-value:  3.3207435990632776e-08


## 5. Lagged Features for Modeling

To enable Bayesian RNN or VAR models to incorporate temporal dependencies and causal effects, we add lagged features. They capture patterns and autocorrelation in the data, helping models learn relationships over time. We drop the NaN values that have been introduced by lagging.

In [61]:
# Added 1-month lag to incorporate temporal dependencies
data['SP500_log_return_lag1'] = data['SP500_log_return'].shift(1)
data['FedFunds_diff_lag1']    = data['FedFunds_diff'].shift(1)

# Added 2-month lag as well
data['SP500_log_return_lag2'] = data['SP500_log_return'].shift(2)
data['FedFunds_diff_lag2']    = data['FedFunds_diff'].shift(2)

In [62]:
data

,FedFundsRate,SP500,SP500_log_return,FedFunds_diff,SP500_log_return_lag1,FedFunds_diff_lag1,SP500_log_return_lag2,FedFunds_diff_lag2
1957-02-28,3.00,43.259998,-0.033192,0.16,NaN,NaN,NaN,NaN
1957-03-31,2.96,44.110001,0.019458,-0.04,-0.033192,0.16,NaN,NaN
1957-04-30,3.00,45.740002,0.036287,0.04,0.019458,-0.04,-0.033192,0.16
1957-05-31,3.00,47.430000,0.036282,0.00,0.036287,0.04,0.019458,-0.04
1957-06-30,3.00,47.369999,-0.001266,0.00,0.036282,0.00,0.036287,0.04
...,...,...,...,...,...,...,...,...
2024-11-30,4.64,6032.379883,0.055720,-0.19,-0.009946,-0.30,0.019996,-0.20
2024-12-31,4.48,5881.629883,-0.025308,-0.16,0.055720,-0.19,-0.009946,-0.30
2025-01-31,4.33,6040.529785,0.026658,-0.15,-0.025308,-0.16,0.055720,-0.19
2025-02-28,4.33,5954.500000,-0.014344,0.00,0.026658,-0.15,-0.025308,-0.16


In [63]:
data.isna().sum()

FedFundsRate             0
SP500                    0
SP500_log_return         0
FedFunds_diff            0
SP500_log_return_lag1    1
FedFunds_diff_lag1       1
SP500_log_return_lag2    2
FedFunds_diff_lag2       2
dtype: int64

In [64]:
# Drop rows with NaN values introduced by lagging
data.dropna(inplace=True)
print(data[['SP500_log_return','SP500_log_return_lag1','FedFunds_diff','FedFunds_diff_lag1']].head())

            SP500_log_return  SP500_log_return_lag1  FedFunds_diff  \
1957-04-30          0.036287               0.019458           0.04   
1957-05-31          0.036282               0.036287           0.00   
1957-06-30         -0.001266               0.036282           0.00   
1957-07-31          0.011335              -0.001266          -0.01   
1957-08-31         -0.057785               0.011335           0.25   

            FedFunds_diff_lag1  
1957-04-30               -0.04  
1957-05-31                0.04  
1957-06-30                0.00  
1957-07-31                0.00  
1957-08-31               -0.01  


In [65]:
data.shape

(816, 8)

## 6. Saving the Preprocessed Data

Once the data has been cleaned, resampled, and transformed, we save the final dataset to a CSV file. This dataset is now ready for further analysis and modeling.

In [66]:
# Save the final dataset to CSV
data.to_csv("financial_timeseries_preprocessed.csv", index=True)

In [67]:
print(data.head(5))
print("...\n")
print(data.tail(5))

            FedFundsRate      SP500  SP500_log_return  FedFunds_diff  \
1957-04-30          3.00  45.740002          0.036287           0.04   
1957-05-31          3.00  47.430000          0.036282           0.00   
1957-06-30          3.00  47.369999         -0.001266           0.00   
1957-07-31          2.99  47.910000          0.011335          -0.01   
1957-08-31          3.24  45.220001         -0.057785           0.25   

            SP500_log_return_lag1  FedFunds_diff_lag1  SP500_log_return_lag2  \
1957-04-30               0.019458               -0.04              -0.033192   
1957-05-31               0.036287                0.04               0.019458   
1957-06-30               0.036282                0.00               0.036287   
1957-07-31              -0.001266                0.00               0.036282   
1957-08-31               0.011335               -0.01              -0.001266   

            FedFunds_diff_lag2  
1957-04-30                0.16  
1957-05-31          

## 7. Summary and Next Steps

In this notebook, we have:
- Collected and merged financial time series data from open sources.
- Cleaned and aligned the data.
- Applied transformations to induce stationarity.
- Verified stationarity with statistical tests.
- Saved the final preprocessed dataset.

Next, we will use this dataset for Bayesian time series modeling and causal intervention analysis.